### Mugrade boilerplate

In [ ]:
### Run this cell to install and import the homework tests
!pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part5_rl_tests.py

import mugrade
import os


from part5_rl_tests import *
os.environ["MUGRADE_HW"] = "Part 5 - Reinforcement Learning"
# Read the Mugrade API key from the environment or from a local .env file.
# Never hard-code the key. See .env.example in the repository root.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
if not os.environ.get("MUGRADE_KEY"):
    print("MUGRADE_KEY is not set. Cells that use @mugrade.submit_tests need it.")

In [ ]:
### Download the necessary files
from huggingface_hub import hf_hub_download
import os

filenames = [
    'llm.d30.pt',
    'tokenizer_50M.bpe',
    'tokenizer_50M_rl.bpe',
    'config.d30.json',
    'config.d30.sft.json',
    'config.d30.rl.json',
    'config.d30.eval.json',
    'gsm8k_train.json',
    'gsm8k_test.json',
]
for f in filenames:
    if not os.path.exists(f):
        hf_hub_download(repo_id="zkolter/llm_speedrun", filename=f, repo_type="dataset", local_dir=".")

### BPE

Copy your BPE implementation from previous parts.  One suggestion: if you do want to run the full RL training runs, you'll want to edit the `decode()` function as shown in class to only decode tokens that are within the length of the vocabulary (with that much generation, there is a substantial probability the LLM generates a token outside this count, since we round the output matrix to the nearest larger multiple of 256).

In [2]:
### BEGIN YOUR CODE
import json
from collections import Counter
import re
from tqdm.auto import tqdm

class BPE:
    def __init__(self, filename=None):
        ### BEGIN YOUR CODE
        if filename is not None:
            with open(filename, "rt") as fobj:
                self.vocab, merge_keys, merge_values, self.special_tokens = json.load(fobj)
                self.merges = {
                    tuple(k): v for k, v in zip(merge_keys, merge_values)
                }
                self.special_tokens = {
                    int(k): v for k, v in self.special_tokens.items()
                }
        else:
            self.vocab = [chr(i) for i in range(256)]
            self.merges = {}
            self.special_tokens = {}
        ### END YOUR CODE

    @staticmethod
    def merge_pair(word, a, b, merged):
        ### BEGIN YOUR CODE
        i = 0
        while i < len(word) - 1:
            if word[i] == a and word[i+1] == b:
                word[i: i+2] = [merged]
            i+=1
        ### END YOUR CODE

    def replace_special_tokens(self, word):
        ### BEGIN YOUR CODE
        for k, v in self.special_tokens.items():
            word = word.replace(v, chr(k))
        return word
        ### END YOUR CODE


    def train(self, text, target_vocab_size):
        ### BEGIN YOUR CODE
        regex_matching = re.findall(r"\s\S+|\s+|\S+", text)
        frequency_counter = Counter(regex_matching)
        frequency_map = {
            self.replace_special_tokens(k): v for k, v in frequency_counter.items() if v > 1
        }

        # Build word_list_frequency_map without defaultdict
        word_list_frequency_map = {}

        for word, freq in frequency_map.items():
            filtered_word = tuple(
                ord(char) for char in word
                if ord(char) not in self.special_tokens
            )

            if filtered_word:
                word_list_frequency_map[filtered_word] = \
                    word_list_frequency_map.get(filtered_word, 0) + freq

        words_as_lists, words_as_lists_frequencies = \
            [list(i) for i in word_list_frequency_map.keys()], list(word_list_frequency_map.values())


        for _ in range(len(self.vocab), target_vocab_size):
            existing_vocab_size = len(self.vocab)
            suitable_merge_pairs = Counter()

            for current_word_as_list, current_word_frequency in \
                zip(words_as_lists, words_as_lists_frequencies):

                for i in range(len(current_word_as_list) - 1):
                    suitable_merge_pairs[(current_word_as_list[i], current_word_as_list[i+1])] +=\
                        current_word_frequency

            # Break condition for testing
            # if len(suitable_merge_pairs) == 0:
            #     break

            symbol_a, symbol_b = suitable_merge_pairs.most_common()[0][0]
            self.vocab.append(self.vocab[symbol_a] + self.vocab[symbol_b])
            self.merges[(symbol_a, symbol_b)] = existing_vocab_size

            for word_as_list in words_as_lists:
                self.merge_pair(word_as_list, symbol_a, symbol_b, existing_vocab_size)

        if len(self.special_tokens) != 0:
            max_special_token = max(self.special_tokens.keys())
            self.vocab += [""] * (max_special_token - len(self.vocab) + 1)
            for k, v in self.special_tokens.items():
                self.vocab[k] = v
        ### END YOUR CODE

    def encode(self, text):
        ### BEGIN YOUR CODE
        tokens = []

        for word in re.findall(r"\s\S+|\s+|\S+", text):
            word_as_list = [ord(char) for char in self.replace_special_tokens(word)]
            # Special tokens are now single code points (e.g. chr(32760)). No merge
            # contains them, so they pass through as their own token IDs.

            if  len(word_as_list) == 0:
                continue

            while True:
                word_symbol_pairs_to_vocab_index_mapping = {}
                for i in range(len(word_as_list) - 1):
                    current_symbol_pair = \
                        (word_as_list[i], word_as_list[i+1])
                    if current_symbol_pair in self.merges:
                        word_symbol_pairs_to_vocab_index_mapping[current_symbol_pair] = \
                            self.merges[current_symbol_pair]

                if len(word_symbol_pairs_to_vocab_index_mapping) == 0:
                    break

                symbol_a, symbol_b = min_symbol_pair = min(
                    word_symbol_pairs_to_vocab_index_mapping,
                    key = word_symbol_pairs_to_vocab_index_mapping.get
                )
                self.merge_pair(word_as_list, symbol_a, symbol_b,
                                word_symbol_pairs_to_vocab_index_mapping[min_symbol_pair])

            tokens += word_as_list

        return tokens
        ### END YOUR CODE

    def decode(self, tokens):
        ### BEGIN YOUR CODE
        return "".join([self.vocab[token] for token in tokens if token < len(self.vocab)])
        ### END YOUR CODE

    def save(self, fname):
        ### BEGIN YOUR CODE
        with open(fname, "wt") as fobj:
            json.dump([
                self.vocab,
                list(self.merges.keys()),
                list(self.merges.values()),
                self.special_tokens
            ], fobj)
        ### END YOUR CODE
### END YOUR CODE

### LLM Architecture

Copy your LLM architecture from the previous assignment on inference.  Only the `generate()` function is tested for this assignment, which you should augment from the previous assignment to include tool calls.
The student notebook has a plain paste field for the previous model code. Implement the separate `generate(self, ...)` function below it; the provided assignment attaches it to your pasted model. Its `self` interface is the same as an instance method.


In [3]:
import torch
import torch.nn.functional as F

import math

def embedding(tokens, weight, dtype):
    ### BEGIN YOUR CODE
    return weight[tokens].to(dtype=dtype)
    ### END YOUR CODE

def linear(x, weight):
    ### BEGIN YOUR CODE
    return x @ weight.type_as(x)
    ### END YOUR CODE

def silu(x):
    ### BEGIN YOUR CODE
    return x / (1 + torch.exp(-x))
    ### END YOUR CODE

def rms_norm(x):
    ### BEGIN YOUR CODE
    x_typed = x.float()
    x_normed = x_typed / (
        (x_typed**2)
            .mean(
                dim = -1,
                keepdim = True,
            )
            .sqrt()
    )
    return x_normed.type_as(x)
    ### END YOUR CODE

def softmax(x):
    ### BEGIN YOUR CODE
    x_numerical_stable = \
        x - x.max(dim = -1, keepdim=True).values
    x_exp = x_numerical_stable.exp()
    return x_exp / (
        x_exp.sum(dim=-1, keepdim=True)
    )
    ### END YOUR CODE

def self_attn(q,k,v,mask):
    ### BEGIN YOUR CODE
    attention_score = softmax(
        q @ k.transpose(-2, -1) / math.sqrt(k.size(-1)) + mask
    )
    return attention_score @ v
    ### END YOUR CODE


class LLM:
    def rope(self, x, pos=0):
        ### BEGIN YOUR CODE
        L = x.shape[-2]
        w1 = self.buffers["rope1"][pos:pos + L]
        w2 = self.buffers["rope2"][pos:pos + L]
        flipped = x.reshape(-1, 2).flip(-1).reshape_as(x)
        return (w1 * x + w2 * flipped).type_as(x)
        ### END YOUR CODE

    def multihead_attn(self, x, layer, mask, pos=0, cache=None):
        ### BEGIN YOUR CODE
        B, L, D = x.shape
        q = linear(x, self.params[f"wq_{layer}"])
        k = linear(x, self.params[f"wk_{layer}"])
        v = linear(x, self.params[f"wv_{layer}"])
        wp = self.params[f"wp_{layer}"]

        q = q.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)

        q = rms_norm(self.rope(q, pos))
        k = rms_norm(self.rope(k, pos))

        if cache is not None:
            c = cache[layer]
            c["k"][:, :, pos:pos + L] = k
            c["v"][:, :, pos:pos + L] = v
            k = c["k"][:, :, :pos + L]
            v = c["v"][:, :, :pos + L]

        attn_output = self_attn(q, k, v, mask)

        return linear(
            attn_output.transpose(1, 2).reshape(B, L, D),
            wp
        )
        ### END YOUR CODE

    def mlp(self, x, layer):
        ### BEGIN YOUR CODE
        return linear(
            silu(
                linear(
                    x,
                    self.params[f"w1_{layer}"]
                )
            ),
            self.params[f"w2_{layer}"]
        )
        ### END YOUR CODE

    def transformer_block(self, x, layer, mask, pos=0, cache=None):
        ### BEGIN YOUR CODE
        x = x + \
            self.multihead_attn(
                rms_norm(x), layer, mask, pos=pos, cache=cache
            )
        return x + self.mlp(
            rms_norm(x), layer
        )
        ### END YOUR CODE

    def __init__(self, config):
        ### BEGIN YOUR CODE
        self.num_layers = config["depth"]
        # JSON configs store the dtype as a string, e.g. "bfloat16".
        dtype = config["dtype"]
        self.dtype = getattr(torch, dtype) if isinstance(dtype, str) else dtype

        vocab_size = config["vocab_size"]
        sequence_len = config["seq_len"]

        hidden_dim = self.num_layers * config["aspect_ratio"]
        self.mlp_dim = hidden_dim * config["mlp_multiple"]

        self.head_dim = config["head_dim"]
        self.num_heads = hidden_dim // self.head_dim

        self.params = {}
        dim_scale, dim_scale_mlp = \
            math.sqrt(2/hidden_dim), math.sqrt(2/self.mlp_dim)

        self.params["embedding"] = torch.randn(
            vocab_size, hidden_dim
        ) * dim_scale
        self.params["output"] = torch.randn(
            hidden_dim, vocab_size
        ) * dim_scale

        for i in range(self.num_layers):
            self.params[f"wq_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wk_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wv_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wp_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"w1_{i}"] = \
                torch.randn(
                    hidden_dim, self.mlp_dim
                ) * dim_scale
            self.params[f"w2_{i}"] = \
                torch.randn(
                    self.mlp_dim, hidden_dim
                ) * dim_scale_mlp

        self.buffers = {}
        mask = torch.full(
            (sequence_len, sequence_len),
            -float("inf"))
        self.buffers["mask"] = \
            torch.triu(mask, diagonal=1)


        freqs = config["rope_theta"] ** (
            (-2 * torch.arange(
                self.head_dim // 2
            ) / self.head_dim)
        )
        theta = torch.outer(
            torch.arange(sequence_len),
            freqs
        )

        cos, sin = torch.cos(theta), torch.sin(theta)
        self.buffers["rope1"] = \
            torch.stack([cos, cos], dim = -1).reshape(
                sequence_len, self.head_dim
            )
        self.buffers["rope2"] = \
            torch.stack([-sin, sin], dim = -1).reshape(
                sequence_len, self.head_dim
            )

        ### END YOUR CODE

    def __call__(self, tokens, pos=0, cache=None):
        ### BEGIN YOUR CODE
        B, L = tokens.shape
        x = rms_norm(
            embedding(
                tokens,
                self.params["embedding"],
                self.dtype
            )
        )

        mask = self.buffers["mask"][pos:pos + L, :pos + L]

        for i in range(self.num_layers):
            x = self.transformer_block(
                x, i, mask, pos=pos, cache=cache
            )

        x_final = \
            rms_norm(x)

        return linear(
            x_final,
            self.params["output"]
        )
        ### END YOUR CODE

    def save(self, filename):
        ### BEGIN YOUR CODE
        torch.save(self.params, filename)
        ### END YOUR CODE

    def load(self, filename):
        ### BEGIN YOUR CODE
        self.params = \
            torch.load(filename, weights_only=True, map_location="cpu")
        ### END YOUR CODE

    @mugrade.submit_tests
    def generate(self, prompts, max_tokens=500, temp=0.7, eos=None, tool_eval=None, bpe=None):
        batch_size = prompts.shape[0]
        prompt_len = prompts.shape[1]

        start_pos, current_pos = 0, prompt_len

        if bpe is not None:
            mask_token_id = bpe.vocab.index("<MASK/>")
            tool_start_id = bpe.vocab.index("<TOOL>")
            tool_end_id = bpe.vocab.index("</TOOL>")
        else:
            mask_token_id = 0

        tokens = torch.cat([
            prompts,
            torch.full((batch_size, max_tokens), mask_token_id, device=prompts.device, dtype=prompts.dtype)
        ], dim=-1)

        max_seq_len = tokens.shape[1]

        kv_cache = [{
            "k": torch.zeros(batch_size, self.num_heads, max_seq_len, self.head_dim, device=prompts.device, dtype=self.dtype),
            "v": torch.zeros(batch_size, self.num_heads, max_seq_len, self.head_dim, device=prompts.device, dtype=self.dtype)
        } for _ in range(self.num_layers)]

        is_finished = [False] * batch_size

        with torch.no_grad():
            while current_pos < max_seq_len and not all(is_finished):
                logits = self(tokens[:, start_pos:current_pos], pos=start_pos, cache=kv_cache)[:, -1]
                probs = F.softmax(logits.float() / temp, dim=-1)

                start_pos = current_pos
                current_pos = current_pos + 1

                for batch_idx in range(batch_size):
                    if tokens[batch_idx, start_pos] == mask_token_id and not is_finished[batch_idx]:
                        sampled_token = torch.multinomial(probs[batch_idx], num_samples=1)
                        tokens[batch_idx, start_pos] = sampled_token

                        if eos is not None:
                            is_finished[batch_idx] = (tokens[batch_idx, start_pos] == eos).item()

                        if tool_eval is not None and tokens[batch_idx, start_pos] == tool_end_id:
                            tool_locs = torch.where(tokens[batch_idx, :start_pos] == tool_start_id)[0]

                            if len(tool_locs) > 0:
                                last_tool_start = tool_locs[-1]
                                tool_call_tokens = tokens[batch_idx, last_tool_start + 1 : start_pos].tolist()

                                response_str = tool_eval(bpe.decode(tool_call_tokens))
                                response_tokens = bpe.encode(f"<RESPONSE>{response_str}</RESPONSE>")

                                response_len = len(response_tokens)

                                if start_pos + response_len + 1 < max_seq_len:
                                    tokens[batch_idx, start_pos + 1 : start_pos + 1 + response_len] = torch.tensor(
                                        response_tokens, device=tokens.device
                                    )

        return tokens

Mugrade: Submitting tests for function generate():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Grader test 5 passed


### LLM Supervised finetuning

The following code augments the tokenizer with the needed tags and downloads and formats the GSM8K dataset.  You don't actually need to run this, as and we provide the fixed tokenizer and dataset files, but it is here for completeness.

In [ ]:
bpe = BPE("tokenizer_50M.bpe")
vocab_size = len(bpe.vocab)
bpe.special_tokens[vocab_size] = "<QUESTION>"
bpe.special_tokens[vocab_size+1] = "</QUESTION>"
bpe.special_tokens[vocab_size+2] = "<THINK>"
bpe.special_tokens[vocab_size+3] = "</THINK>"
bpe.special_tokens[vocab_size+4] = "<TOOL>"
bpe.special_tokens[vocab_size+5] = "</TOOL>"
bpe.special_tokens[vocab_size+6] = "<RESPONSE>"
bpe.special_tokens[vocab_size+7] = "</RESPONSE>"
bpe.special_tokens[vocab_size+8] = "<ANSWER>"
bpe.special_tokens[vocab_size+9] = "</ANSWER>"
bpe.special_tokens[vocab_size+10] = "<MASK/>"

bpe.vocab += [""]*(max(bpe.special_tokens.keys())-len(bpe.vocab)+1)
for k,v in bpe.special_tokens.items():
    bpe.vocab[k] = v
bpe.save("tokenizer_50M_rl.bpe")

from datasets import load_dataset
import random
max_seq_len = 512+1 # max actual GSM8K size is 400, so add additional padding
mask_token = bpe.vocab.index("<MASK/>")

for split in ["train", "test"]:
    tokens = []
    for example in load_dataset("openai/gsm8k", "main", split=split):
        question = example["question"]
        think, answer = example["answer"].split("\n#### ")
        for toolcall in re.findall(r"<<.+?>>", think):
            call, resp = toolcall[2:-2].split("=")
            think = think.replace(toolcall, f"<TOOL>{call}</TOOL><RESPONSE>{resp}</RESPONSE>")
        text = f"<DOCUMENT><QUESTION>{question}</QUESTION><THINK>{think}</THINK><ANSWER>{answer.replace(",","")}</ANSWER></DOCUMENT>"
        tokens.append({"tokens":bpe.encode(text), "text":text})
        tokens[-1]["tokens"] += [mask_token]*(max_seq_len-len(tokens[-1]["tokens"]))
    random.seed(42)
    random.shuffle(tokens)
    with open(f"gsm8k_{split}.json", "wt") as f:
        json.dump(tokens, f)

Implement the functions needed for supervised finetuning.  You will need to augment the cross entropy loss function from previous assignments to allow for a mask token (where loss is not applied) and a weight term, that can weight each element of the computed loss.  We'll take the convention you should still normalize by the total number of non-masked tokens (regardless of the weight term).  Implement Adam exactly like in the previous portions but _without_ a learning rate schedule.  Finally, implement supervised finetuning.  As in class, you should randomly pad inputs with some number of `<MASK/>` tokens, so that the model learns to ignore mask tokens at the start of a sequence (so that you can later do parallel generation over multiple prompts).


In [12]:
import random
import wandb
import time
from array import array
import os

@mugrade.submit_tests
def cross_entropy_loss(logits, y, mask_token=-1, weights=1):
    ### BEGIN YOUR CODE
    logits_normed = logits - logits.max(dim = -1, keepdim = True)[0]
    loss = -logits_normed.take_along_dim(y.unsqueeze(-1), dim = -1)
    loss += logits_normed.exp().sum(dim = -1, keepdim = True).log()
    return (
        (weights * loss)[y != mask_token].sum() / \
        (y != mask_token).float().sum()
    )

    ### END YOUR CODE

class Adam:
    @mugrade.submit_tests
    def __init__(self, params, lr=1e-3, betas = (0.9, 0.95), eps=1e-5, weight_decay=0.0):
        ### BEGIN YOUR CODE
        self.lr = lr
        self.betas = betas
        self.eps = eps
        self.weight_decay = weight_decay
        self.params = params

        self.t = 1

        self.u = {k: torch.zeros_like(v) for k,v in self.params.items()}
        self.v = {k: tensor.clone().detach() for k, tensor in self.u.items()}

        for param in self.params.values():
            param.requires_grad_()
        ### END YOUR CODE

    @mugrade.submit_tests
    def step(self):
        ### BEGIN YOUR CODE
        with torch.no_grad():
            for k,p in self.params.items():
                self.u[k] = self.betas[0]*self.u[k] + (1-self.betas[0])*p.grad
                self.v[k] = self.betas[1]*self.v[k] + (1-self.betas[1])*p.grad**2

                u_hat = self.u[k] / (1 - self.betas[0]**self.t)
                v_hat = self.v[k] / (1 - self.betas[1]**self.t)

                p *= (1 - self.lr * self.weight_decay)
                p -= self.lr * u_hat / (v_hat.sqrt() + self.eps)

                p.grad.zero_()
            self.t += 1
        ### END YOUR CODE

@mugrade.submit_tests
def finetune_llm_sft(rank, nccl_uid, config):
    ### BEGIN YOUR CODE
    torch.cuda.set_device(rank)

    with open(config["llm_config"]) as fobj:
        llm_config = json.load(fobj)

    llm = LLM(llm_config)
    llm.load(config["infilename"])

    for k in llm.params:
        llm.params[k] = llm.params[k].detach().cuda()
    for k in llm.buffers:
        llm.buffers[k] = llm.buffers[k].cuda()

    comm = torch.cuda.nccl.init_rank(config["num_gpus"], nccl_uid, rank)
    opt = Adam(llm.params, lr=config["lr"])

    with open(config["datafile"]) as f:
        dataset = json.load(f)

    bpe = BPE(config["tokenizer"])
    mask_token = bpe.vocab.index("<MASK/>")

    for _ in range(config["epochs"]):
        for i in range(0, len(dataset), config["batch_size"]):
            local_batch_size = (min(config["batch_size"], len(dataset) - i) + config["num_gpus"] - 1) // config["num_gpus"]
            data = dataset[i + rank * local_batch_size : i + (rank + 1) * local_batch_size]

            # sft loss
            tokens = [t["tokens"] for t in data]
            for j in range(len(tokens)):
                shift = random.randint(0, config["seq_len"] - tokens[j].index(mask_token) - 1)
                tokens[j] = tokens[j][-shift:] + tokens[j][:-shift]
            tokens = torch.tensor(tokens).cuda()
            logits = llm(tokens[:, :-1]).float()
            loss = cross_entropy_loss(logits, tokens[:, 1:], mask_token=mask_token)
            loss.backward()

            for p in llm.params.values():
                torch.cuda.nccl.all_reduce([p.grad], comms=[comm])
                p.grad /= config["num_gpus"]
            opt.step()

            torch.cuda.nccl.all_reduce([loss], comms=[comm])

    if rank == 0:
        llm.save(config["filename"])
    ### END YOUR CODE


Mugrade: Submitting tests for function cross_entropy_loss():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Mugrade: Submitting tests for function __init__():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Grader test 5 passed
Mugrade: Submitting tests for function step():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Grader test 5 passed
Mugrade: Submitting tests for function finetune_llm_sft():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Grader test 5 passed
Grader test 6 passed
Grader test 7 passed
Grader test 8 passed
Grader test 9 passed
Grader test 10 passed
Grader test 11 passed
Grader test 12 passed


Once you have implemented the above, you should be able to finetune a model with the following code (not tested, only if you have a GPU and want to run it).

In [ ]:
from joblib import Parallel, delayed
import os
os.environ["NCCL_NVLS_ENABLE"] = "0"

with open("config.d30.sft.json", "rt") as f: config = json.load(f)
nccl_uid = torch.cuda.nccl.unique_id()
Parallel(n_jobs = config["num_gpus"], backend="loky")(
    delayed(finetune_llm_sft)(i, nccl_uid, config) for i in range(config["num_gpus"])
)

### Evaluating task performance

Once you have finetuned the model, you need to now use generation and tool calling to actually generate responses to problems and evaluate the performance.  Write the functions to call the tool (as we did in class, just by calling `eval()`), to extract answers from a response or ground truth, to grade correctness and correct formatting of responses, to evaluate a set of answers, and finally to evaluate a full dataset.

In [13]:
import ast
import operator

# The calculator tool evaluates text that the model generates. Python's eval()
# would run arbitrary code, so this evaluator accepts only numeric literals,
# + - * / // % **, unary signs, and parentheses.
_ARITH_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod, ast.Pow: operator.pow,
    ast.UAdd: operator.pos, ast.USub: operator.neg,
}
_MAX_EXPONENT = 100  # Stops inputs such as 9**9**9 from exhausting memory.

def _safe_arith(node):
    if isinstance(node, ast.Expression):
        return _safe_arith(node.body)
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ARITH_OPS:
        return _ARITH_OPS[type(node.op)](_safe_arith(node.operand))
    if isinstance(node, ast.BinOp) and type(node.op) in _ARITH_OPS:
        left, right = _safe_arith(node.left), _safe_arith(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > _MAX_EXPONENT:
            raise ValueError("exponent too large")
        return _ARITH_OPS[type(node.op)](left, right)
    raise ValueError(f"unsupported expression: {type(node).__name__}")

@mugrade.submit_tests
def tool_eval(text):
    ### BEGIN YOUR CODE
    try:
        response = round(_safe_arith(ast.parse(text.strip(), mode="eval")), 2)
        if int(response) == response:
            response = int(response)
    except Exception:
        response = ""
    return response
    ### END YOUR CODE

@mugrade.submit_tests
def extract_answer(text):
    ### BEGIN YOUR CODE
    try:
        return int(re.search(r"<ANSWER>(.*?)</ANSWER>", text).group(1))
    except:
        return None
    ### END YOUR CODE

@mugrade.submit_tests
def grade(responses, ground_truths):
    ### BEGIN YOUR CODE
    response_answers = [extract_answer(text) for text in responses]
    correct_answers = [extract_answer(text) for text in ground_truths]
    correct = [a == b for a, b in zip(response_answers, correct_answers)]
    formatted = [a is not None for a in response_answers]
    return correct, formatted
    ### END YOUR CODE

@mugrade.submit_tests
def eval_examples(llm, examples, bpe, tool_eval, k, seq_len):
    ### BEGIN YOUR CODE
    mask_token = bpe.vocab.index("<MASK/>")
    think_token = bpe.vocab.index("<THINK>")
    answer_token = bpe.vocab.index("</ANSWER>")

    prompts = [e["tokens"][:e["tokens"].index(think_token)+1] for e in examples] * k
    answers = [e["text"] for e in examples] * k
    max_len = max(len(p) for p in prompts)
    for i in range(len(prompts)):
        prompts[i] = [mask_token]*(max_len - len(prompts[i])) + prompts[i]
    prompts = torch.tensor(prompts).cuda()

    tokens = llm.generate(prompts, max_tokens=seq_len-max_len, eos=answer_token,
        tool_eval=tool_eval, bpe=bpe)
    responses = [bpe.decode(t.tolist()).replace("<MASK/>", "") for t in tokens]

    return tokens, grade(responses, answers)
    ### END YOUR CODE


@mugrade.submit_tests
def eval_dataset(rank, config):
    ### BEGIN YOUR CODE
    with open(config["llm_config"], "rt") as f: llm_config = json.load(f)
    llm = LLM(llm_config)
    llm.load(config["filename"])
    torch.cuda.set_device(rank)
    for k in llm.params: llm.params[k] = llm.params[k].detach().cuda()
    for k in llm.buffers: llm.buffers[k] = llm.buffers[k].cuda()

    with open(config["datafile"], "rt") as f: dataset = json.load(f)
    bpe = BPE(config["tokenizer"])

    pass_at_1 = 0
    pass_at_k = 0
    valid = 0

    n = config["batch_size"] // config["k"]
    for i in range(0, len(dataset), n):
        local_batch_size = min(n, len(dataset)-i) // config["num_gpus"]
        data = dataset[i+rank*local_batch_size:i+(rank+1)*local_batch_size]

        _, (correct, formatted) = eval_examples(llm, data, bpe, tool_eval,
            config["k"], config["seq_len"])
        pass_at_1 += sum(correct)
        pass_at_k += sum([any(correct[i::local_batch_size]) for i in range(local_batch_size)])
        valid += sum(formatted)
        if rank == 0:
            print(i)

    m = len(dataset) - len(dataset) % config["num_gpus"]
    return pass_at_1 / (m*config["k"]), pass_at_k / m, valid / (m*config["k"])
    ### END YOUR CODE

Mugrade: Submitting tests for function tool_eval():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Mugrade: Submitting tests for function extract_answer():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Mugrade: Submitting tests for function grade():
Mugrade: Submitting tests for function eval_examples():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Grader test 5 passed
Grader test 6 passed
Mugrade: Submitting tests for function eval_dataset():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Grader test 5 passed


After this implementation, the following code will evaluate the sft-trained model.

In [ ]:
from joblib import Parallel, delayed

with open("config.d30.eval.json", "rt") as f: config = json.load(f)
config["filename"] = "llm.d30.sft.pt"
out = Parallel(n_jobs = config["num_gpus"], backend="loky")(
    delayed(eval_dataset)(i, config) for i in range(config["num_gpus"])
)
pass_at_1, pass_at_k, valid_at_1 = [sum(o[i] for o in out) for i in range(3)]
print(f"Pass@1: {pass_at_1}, Pass@{config["k"]}: {pass_at_k}, Valid: {valid_at_1}")

### Training with RL

Finally, adapt the SFT training function to include RL training.  As demonstrated in class, given all the functions above, this actually represents a fairly straightforward modification of the SFT finetuning code.


In [19]:
@mugrade.submit_tests
def finetune_llm_rl(rank, nccl_uid, config):
    ### BEGIN YOUR CODE
    with open(config["llm_config"], "rt") as f: llm_config = json.load(f)
    llm = LLM(llm_config)
    llm.load(config["infilename"])
    torch.cuda.set_device(rank)
    for k in llm.params: llm.params[k] = llm.params[k].detach().cuda()
    for k in llm.buffers: llm.buffers[k] = llm.buffers[k].cuda()

    comm = torch.cuda.nccl.init_rank(config["num_gpus"], nccl_uid, rank)

    opt = Adam(llm.params, lr=config["lr"])
    with open(config["datafile"], "rt") as f: dataset = json.load(f)
    bpe = BPE(config["tokenizer"])
    mask_token = bpe.vocab.index("<MASK/>")

    n = config["batch_size"] // config["k"]
    for _ in range(config["epochs"]):
        for i in range(0, len(dataset), n):
            local_batch_size = min(n, len(dataset)-i) // config["num_gpus"]
            data = dataset[i+rank*local_batch_size:i+(rank+1)*local_batch_size]

            tokens = [t["tokens"] for t in data]
            for j in range(len(tokens)):
                shift = random.randint(0, config["seq_len"]-tokens[j].index(mask_token)-1)
                tokens[j] = tokens[j][-shift:] + tokens[j][:-shift]
            tokens = torch.tensor(tokens).cuda()
            logits = llm(tokens[:,:-1]).float()
            sft_loss = config["sft_loss_weight"]*cross_entropy_loss(logits, tokens[:,1:], mask_token=mask_token)
            sft_loss.backward()

            tokens, (correct, formatted) = eval_examples(llm, data, bpe, tool_eval, config["k"], config["seq_len"]+1)
            reward = (1.0*torch.tensor(correct) + 0.05*torch.tensor(formatted)).cuda()
            adv = reward.reshape(config["k"], local_batch_size)
            adv = ((adv - adv.mean(dim=0, keepdim=True))/(adv.std(dim=0, keepdim=True)+1e-5)).reshape(-1)
            logits = llm(tokens[:,:-1]).float()

            rl_loss = cross_entropy_loss(logits, tokens[:,1:], mask_token=mask_token, weights=adv[:,None,None])
            rl_loss.backward()

            for p in llm.params.values():
                torch.cuda.nccl.all_reduce([p.grad], comms=[comm])
                p.grad /= config["num_gpus"]
            opt.step()

            torch.cuda.nccl.all_reduce([reward], comms=[comm])

    if rank == 0:
        llm.save(config["filename"])
    ### END YOUR CODE

Mugrade: Submitting tests for function finetune_llm_rl():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed
Grader test 4 passed
Grader test 5 passed
Grader test 6 passed
Grader test 7 passed
Grader test 8 passed
Grader test 9 passed
Grader test 10 passed
Grader test 11 passed
Grader test 12 passed
Grader test 13 passed
Grader test 14 passed


After this code, the following cells will RL train the model then evaluate it.

In [ ]:
from joblib import Parallel, delayed
import os
os.environ["NCCL_NVLS_ENABLE"] = "0"

with open("config.d30.rl.json", "rt") as f: config = json.load(f)
nccl_uid = torch.cuda.nccl.unique_id()

Parallel(n_jobs = config["num_gpus"], backend="loky")(
    delayed(finetune_llm_rl)(i, nccl_uid, config) for i in range(config["num_gpus"])
)

In [ ]:
with open("config.d30.eval.json", "rt") as f: config = json.load(f)
out = Parallel(n_jobs = config["num_gpus"], backend="loky")(
    delayed(eval_dataset)(i, config) for i in range(config["num_gpus"])
)
pass_at_1, pass_at_k, valid_at_1 = [sum(o[i] for o in out) for i in range(3)]
print(f"Pass@1: {pass_at_1}, Pass@{config["k"]}: {pass_at_k}, Valid: {valid_at_1}")